## Import crucial libraries

In [ ]:
import shutil
import random
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import os
from tqdm.notebook import tqdm

## Download dataset

In [ ]:
!pip install kagglehub #install library

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


# download latest version
path = kagglehub.dataset_download("joebeachcapital/defungi")

print("Path to dataset files:", path) # Prints the path of extracted files

# Assuming the dataset contains a 'data.csv' file, access it using the path:
# import pandas as pd
# df = pd.read_csv(path + '/data.csv')



## Splitting Dataset

In [ ]:
source_root = path
target_root = "/root/fungi_split"
train_ratio = 0.8

for class_name in ["H1", "H2", "H3", "H5", "H6"]:
    src_dir = os.path.join(source_root, class_name)
    all_images = [img for img in os.listdir(src_dir) if img.endswith(".jpg")]
    random.shuffle(all_images)
    split_point = int(train_ratio * len(all_images))
    train_imgs = all_images[:split_point]
    test_imgs = all_images[split_point:]

    # train folder
    train_class_dir = os.path.join(target_root, "train", class_name)
    os.makedirs(train_class_dir, exist_ok=True)
    for img in train_imgs:
        shutil.copy(os.path.join(src_dir, img), os.path.join(train_class_dir, img))

    # test folder
    test_class_dir = os.path.join(target_root, "test", class_name)
    os.makedirs(test_class_dir, exist_ok=True)
    for img in test_imgs:
        shutil.copy(os.path.join(src_dir, img), os.path.join(test_class_dir, img))

print("Split complete! Dataset now in /root/fungi_split")

In [ ]:
train_dir = "/root/fungi_split/train"
test_dir = "/root/fungi_split/test"

# count images in train set
train_count = sum(len(files) for _, _, files in os.walk(train_dir))
# Count images in test set
test_count = sum(len(files) for _, _, files in os.walk(test_dir))

total_count = train_count + test_count

print(f"Train Set: {train_count} images")
print(f"Test Set: {test_count} images")
print(f"Total: {total_count} images")


In [ ]:
# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===============================
# 1. Image Preprocessing
# ===============================
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize to [-1, 1]
])

# ===============================
# 2. Load Dataset
# ===============================
dataset_path = "/root/fungi_split"

train_data = datasets.ImageFolder(os.path.join(dataset_path, "train"), transform=transform)
test_data = datasets.ImageFolder(os.path.join(dataset_path, "test"), transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = train_data.classes
num_classes = len(class_names)
print("Classes:", class_names)

# ===============================
# 3. CNN Model Definition
# ===============================
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1)),  # ensures fixed size
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.model(x)

model = CNNModel(num_classes).to(device)
print(model)

# ===============================
# 4. Loss and Optimizer
# ===============================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ===============================
# 5. Training the Model
# ===============================
num_epochs = 15
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # Training
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    print(f"Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_acc:.4f}")

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    val_epoch_loss = val_loss / len(test_loader)
    val_epoch_acc = val_correct / val_total
    val_losses.append(val_epoch_loss)
    val_accuracies.append(val_epoch_acc)
    print(f"Val Loss: {val_epoch_loss:.4f} | Val Accuracy: {val_epoch_acc:.4f}")

# ===============================
# 6. Evaluation and Metrics
# ===============================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

# Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ===============================
# 7. Training Visualization
# ===============================
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='o')
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_accuracies, label='Train Acc', marker='o')
plt.plot(val_accuracies, label='Val Acc', marker='o')
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()
